In [ ]:
import os
import json
import glob
import gc
import time
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

DATA_ROOT = "/kaggle/input/datasets/michaildarm/zeroer-benchamarks/"
OUTPUT_ROOT = "/kaggle/working/embeddings"

# Which passes to run below. Mirrors gpu-heavy-processes.ipynb and
# datasets/embeddings/{record,attribute}/ on the target machine (data.py:
# RECORD_DIR, ATTR_DIR).
ENCODE_RECORD     = True    # whole-record: embeddings/record/<dataset>/
ENCODE_ATTRIBUTES = True    # per-attribute: embeddings/attribute/<dataset>/

RECORD_ROOT = os.path.join(OUTPUT_ROOT, "record")
ATTR_ROOT   = os.path.join(OUTPUT_ROOT, "attribute")

os.makedirs(RECORD_ROOT, exist_ok=True)
os.makedirs(ATTR_ROOT, exist_ok=True)

# One line per encode() call that actually ran (cache hits aren't timed --
# nothing was created).
TIMING_LOG = os.path.join(OUTPUT_ROOT, "embedding_times.txt")

In [ ]:
E5_INSTRUCT = "Instruct: Retrieve semantically similar text.\nQuery: "

PLM_MODELS = {
    "minilm6":       ("sentence-transformers/all-MiniLM-L6-v2", ""),
    "minilm12":      ("sentence-transformers/all-MiniLM-L12-v2", ""),
    "distilroberta": ("sentence-transformers/all-distilroberta-v1", ""),
    "mpnet":         ("sentence-transformers/all-mpnet-base-v2", ""),
    "gtr_t5_large":  ("sentence-transformers/gtr-t5-large", ""),
    "bge_large":     ("BAAI/bge-large-en-v1.5",
                      "Represent this sentence for searching relevant passages: "),
    "gte_large":     ("thenlper/gte-large", ""),
    "e5_large_v2":   ("intfloat/e5-large-v2", "query: "),
    "st5_xl":        ("sentence-transformers/sentence-t5-xl", ""),
    "e5_instruct":   ("intfloat/multilingual-e5-large-instruct", E5_INSTRUCT),
}

BATCH_SIZE = 128
MAX_LENGTH = 512


In [ ]:
# Per-dataset attribute allow-list, mirroring data.py DATASETS /
# gpu-heavy-processes.ipynb. `id` never feeds the model (on dblp_scholar the
# left ids encode authors/venue/year while the right ids are opaque hashes),
# and fodors_zagats' `class` is the ground-truth cluster label -- all 112
# matched pairs share it, and no other cross-table pair does.
ATTRIBUTES = {
    "fodors_zagats":         ["name", "addr", "city", "phone"],
    "dblp_acm":              ["title", "authors", "venue", "year"],
    "dblp_scholar":          ["title", "authors", "venue", "year"],
    "abt_buy":               ["name", "description", "price"],
    "amazon_googleproducts": ["title", "description", "manufacturer", "price"],
}

# Right-table renames applied before attribute selection (utils.py rename_2).
RENAME_RIGHT = {
    "amazon_googleproducts": {"name": "title"},
}

datasets = []

for root, dirs, files in os.walk(DATA_ROOT):

    if "left.csv" in files and "right.csv" in files:

        name = os.path.basename(root)

        if name not in ATTRIBUTES:
            raise KeyError(f"no attribute list for dataset {name!r}")

        datasets.append({
            "name": name,
            "path": root,
            "left": os.path.join(root, "left.csv"),
            "right": os.path.join(root, "right.csv"),
        })

print(f"Found {len(datasets)} datasets")


def dataframe_to_text(df, attributes):

    missing = [c for c in attributes if c not in df.columns]

    if missing:
        raise KeyError(f"missing columns {missing}; have {list(df.columns)}")

    texts = []

    for row in df[attributes].itertuples(index=False):

        parts = []
        for c, v in zip(attributes, row):

            if pd.isna(v):
                continue

            v = str(v).strip()

            if not v:
                continue

            parts.append(f"{c}: {v}")

        texts.append(" | ".join(parts))

    return texts

In [ ]:
def embed_texts(
    texts,
    model,
    prefix="",
    batch_size=BATCH_SIZE,
):
    if prefix:
        texts = [prefix + t for t in texts]

    pool = model.start_multi_process_pool(
        target_devices=["cuda:0", "cuda:1"]
    )

    emb = model.encode(
        texts,
        pool=pool,
        batch_size=batch_size,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    model.stop_multi_process_pool(pool)

    return emb


def timed_embed_texts(texts, model, log_fields, **kwargs):
    """embed_texts(), timed end-to-end and appended to TIMING_LOG.

    ``log_fields`` identifies the call (phase, dataset, model, side, ...);
    n_texts and seconds are added automatically. ``model.encode(pool=...)``
    blocks until the worker processes return, so the wall-clock elapsed here
    already covers the multi-GPU encode -- no extra synchronize needed.
    """
    t0 = time.perf_counter()
    vectors = embed_texts(texts, model, **kwargs)
    elapsed = time.perf_counter() - t0

    fields = dict(log_fields, n_texts=len(texts), seconds=f"{elapsed:.3f}")
    line = " | ".join(f"{k}={v}" for k, v in fields.items())
    ts = time.strftime("%Y-%m-%dT%H:%M:%S")
    with open(TIMING_LOG, "a") as f:
        f.write(f"{ts} | {line}\n")
    print(f"  [{elapsed:.1f}s] {line}")

    return vectors

In [ ]:
if ENCODE_RECORD:
    for ds in datasets:

        ds_name = ds["name"]
        print("\n" + "=" * 100)
        print("DATASET:", ds_name)

        out_dir = os.path.join(RECORD_ROOT, ds_name)
        os.makedirs(out_dir, exist_ok=True)

        attributes = ATTRIBUTES[ds_name]

        left_df = pd.read_csv(ds["left"])
        right_df = pd.read_csv(ds["right"])

        if ds_name in RENAME_RIGHT:
            right_df = right_df.rename(columns=RENAME_RIGHT[ds_name])

        left_texts = dataframe_to_text(left_df, attributes)
        right_texts = dataframe_to_text(right_df, attributes)

        for model_name, (model_path, prefix) in PLM_MODELS.items():

            short_name = model_name.replace("/", "_")

            print("\n" + "-" * 80)
            print("Loading model:", model_name)

            left_cache_path = os.path.join(out_dir, f"{short_name}_left.npy")
            right_cache_path = os.path.join(out_dir, f"{short_name}_right.npy")

            if os.path.exists(left_cache_path) and os.path.exists(right_cache_path):
                print("Cache exists → skipping")
                continue

            model = SentenceTransformer(model_path)

            if os.path.exists(left_cache_path):
                left_emb = np.load(left_cache_path)
            else:
                left_emb = timed_embed_texts(
                    left_texts, model,
                    log_fields=dict(phase="record", dataset=ds_name,
                                     model=model_name, side="left"),
                    prefix=prefix)
                np.save(left_cache_path, left_emb)

            if os.path.exists(right_cache_path):
                right_emb = np.load(right_cache_path)
            else:
                right_emb = timed_embed_texts(
                    right_texts, model,
                    log_fields=dict(phase="record", dataset=ds_name,
                                     model=model_name, side="right"),
                    prefix=prefix)
                np.save(right_cache_path, right_emb)

            print("Shape:", left_emb.shape, right_emb.shape)

            del model
            del left_emb
            del right_emb
            gc.collect()
            torch.cuda.empty_cache()
else:
    print("ENCODE_RECORD is False -> skipping whole-record pass")

In [ ]:
# --- Arm B2 config ----------------------------------------------------------
# All ten PLM_MODELS -- unlike the 8B-14B LLM backbones in
# gpu-heavy-processes.ipynb, these are cheap enough not to need a top-3 cut.
ATTR_MODELS = list(PLM_MODELS)

# Attribute values are far shorter than a serialised record, so the batch can
# be bigger than BATCH_SIZE.
ATTR_BATCH_SIZE = 256

print(f"{len(ATTR_MODELS)} models x per-attribute passes")

In [ ]:
def attribute_column(df, attribute):
    """(unique_texts, inverse, present) for one attribute of one table.

    Serialisation matches ``dataframe_to_text`` with a single-element attribute
    list -- the ``"{attr}: {value}"`` prefix is kept, so the encoder still sees
    which field it is reading.

    Two things the whole-record path never had to handle:

    * **Missing values.** A record with no value for this attribute cannot be
      embedded -- an empty string yields an arbitrary vector whose cosine
      against anything is noise. Those rows are excluded from the encode and
      left as an exact zero row, with ``present`` recording which. Downstream
      the feature must be forced to 0 for any pair where either side is absent,
      matching ``zeroer_features``' documented Magellan convention (NaN in
      either value -> NaN feature -> 0 after fillna).
    * **Duplicates.** Single attributes repeat heavily where whole records do
      not (venue, year, city, manufacturer, price). Encoding unique values only
      and scattering back cuts the corpus substantially with identical output.
    """
    if attribute not in df.columns:
        raise KeyError(f"missing column {attribute!r}; have {list(df.columns)}")

    s = df[attribute].astype("string").str.strip()
    present = (df[attribute].notna() & s.notna() & (s != "")).to_numpy()

    texts = (attribute + ": " + s[present]).to_numpy()
    uniq, inverse = np.unique(texts, return_inverse=True)   # sorted uniques
    return uniq.tolist(), inverse, present


def scatter(vectors, inverse, present, n_rows):
    """Unique-value vectors -> one row per record; absent rows stay zero."""
    full = np.zeros((n_rows, vectors.shape[1]), dtype=vectors.dtype)
    full[present] = vectors[inverse]
    return full

In [ ]:
if ENCODE_ATTRIBUTES:
    for ds in datasets:

        ds_name = ds["name"]
        attributes = ATTRIBUTES[ds_name]

        out_dir = os.path.join(ATTR_ROOT, ds_name)
        os.makedirs(out_dir, exist_ok=True)

        print("\n" + "=" * 100)
        print("DATASET:", ds_name, "| attributes:", attributes)

        left_df = pd.read_csv(ds["left"])
        right_df = pd.read_csv(ds["right"])
        if ds_name in RENAME_RIGHT:
            right_df = right_df.rename(columns=RENAME_RIGHT[ds_name])
        sides = {"left": left_df, "right": right_df}

        # ---- presence masks: model-independent, written once -------------------
        presence_path = os.path.join(out_dir, "presence.npz")
        if not os.path.exists(presence_path):
            np.savez_compressed(presence_path, **{
                f"{a}_{side}": attribute_column(df, a)[2]
                for side, df in sides.items() for a in attributes})
            print("Saved presence masks:", presence_path)

        for model_name in ATTR_MODELS:

            model_path, prefix = PLM_MODELS[model_name]
            short_name = model_name.replace("/", "_")

            todo = [(a, side) for a in attributes for side in ("left", "right")
                    if not os.path.exists(
                        os.path.join(out_dir, f"{short_name}_{a}_{side}.npy"))]

            if not todo:
                print(f"  {short_name}: all {len(attributes)*2} files cached -> skip")
                continue

            print("\n" + "-" * 80)
            print("Loading model:", model_name, f"({len(todo)} passes)")

            model = SentenceTransformer(model_path)

            for attribute, side in todo:

                df = sides[side]
                uniq, inverse, present = attribute_column(df, attribute)

                print(f"  {attribute:12s} {side:5s} "
                      f"{len(df):6d} rows | {len(uniq):6d} unique | "
                      f"{int((~present).sum()):5d} absent")

                vectors = timed_embed_texts(
                    uniq, model,
                    log_fields=dict(phase="attribute", dataset=ds_name,
                                     model=model_name, attribute=attribute, side=side),
                    prefix=prefix,
                    batch_size=ATTR_BATCH_SIZE)

                full = scatter(np.asarray(vectors), inverse, present, len(df))

                path = os.path.join(out_dir, f"{short_name}_{attribute}_{side}.npy")
                np.save(path, full)

                del vectors, full
                gc.collect()

            del model
            gc.collect()
            torch.cuda.empty_cache()
else:
    print("ENCODE_ATTRIBUTES is False -> skipping per-attribute pass")

In [ ]:
# --- verify -----------------------------------------------------------------
# Row alignment with the source CSVs is the one invariant the downstream
# feature builder cannot check for itself, and a zero row must mean "absent",
# never "encoded badly".
for ds in datasets:

    ds_name = ds["name"]
    out_dir = os.path.join(ATTR_ROOT, ds_name)
    if not os.path.isdir(out_dir):
        continue

    presence = np.load(os.path.join(out_dir, "presence.npz"))
    n_rows = {"left": len(pd.read_csv(ds["left"])),
              "right": len(pd.read_csv(ds["right"]))}

    for model_name in ATTR_MODELS:
        short_name = model_name.replace("/", "_")

        for attribute in ATTRIBUTES[ds_name]:
            for side in ("left", "right"):

                path = os.path.join(out_dir, f"{short_name}_{attribute}_{side}.npy")
                if not os.path.exists(path):
                    print(f"MISSING {path}")
                    continue

                vec = np.load(path, mmap_mode="r")
                mask = presence[f"{attribute}_{side}"]

                assert len(vec) == n_rows[side], (
                    f"{path}: {len(vec)} rows vs {n_rows[side]} in {side}.csv")

                zero = ~np.asarray(vec).any(axis=1)
                assert (zero == ~mask).all(), (
                    f"{path}: zero rows disagree with the presence mask "
                    f"({zero.sum()} zero vs {(~mask).sum()} absent)")

    print(f"{ds_name}: OK")